# bispectrosa: a tour

Most common sound summary statistics (spectrograms, or the MFCCs built from them) come down to *which*
frequencies are present and how loud. They cannot tell whether those frequencies just happen to
occur at the same time, or are related, generated together by one underlying process.

One of the simplest quantities that can is the [**bispectrum**](https://en.wikipedia.org/wiki/Bispectrum),
the Fourier transform of the third-order cumulant (the familiar power spectrum is second order). It responds
when three frequencies $f_1$, $f_2$, and their sum $f_1{+}f_2$ stay in step: locked together because one
process generated them, rather than lining up by chance.

`bispectrosa` computes it with a librosa-style API: `bs.mel_bispectrogram(y, sr)`.

For the math behind it, see [`../docs/theory.md`](../docs/theory.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import librosa                 # only used to load the example recordings
import bispectrosa as bs

In [ ]:
SR = 16000                     # sample rate everything runs at (Hz)
N_FFT = 400                    # STFT frame length; matches the library default
freqs = np.fft.rfftfreq(N_FFT, 1 / SR)         # centre frequency of each STFT bin (Hz)

def hz_to_bin(f):
    # nearest STFT bin index for a frequency in Hz
    return np.clip(np.round(np.asarray(f) / SR * N_FFT).astype(int), 0, N_FFT // 2)

# A small helper to load a librosa example clip, resampled to SR, first 4 seconds.
def load(key): return librosa.load(librosa.example(key), sr=SR, mono=True, duration=4.0)[0]

In [ ]:
# The one call to remember: the mel bispectrogram, a feature shaped like a spectrogram
# (the analogue of librosa.feature.melspectrogram, one order up). The rest of this notebook
# shows what it measures and why it works.
feat = bs.mel_bispectrogram(load('trumpet'), sr=SR)
feat.shape                                     # (49 coefficients, n_frames)

In [ ]:
# Plot styling, shared by every figure below.
plt.rcParams.update({'font.size': 10, 'axes.linewidth': 0.8, 'mathtext.fontset': 'cm'})
kHz = FuncFormatter(lambda x, _: f'{x/1000:g}')   # tick formatter: label Hz axes in kHz
kticks = [1000, 2000, 3000, 4000]                 # labelled frequency ticks (1 2 3 4 kHz)
kminor = [500, 1500, 2500, 3500]                  # unlabelled minor ticks between them

def freq_axis(ax, axis='x'):
    # 1-4 kHz labelled ticks, minor ticks at the halves, labels in kHz
    set_ticks = ax.set_xticks if axis == 'x' else ax.set_yticks
    set_ticks(kticks); set_ticks(kminor, minor=True)
    getattr(ax, f'{axis}axis').set_major_formatter(kHz)

## 1. What the bispectrum measures: two controlled examples

What does "correlated frequencies" mean? Take the simplest possible signal: sine components
at $f_1$, $f_2$ and their sum $f_1{+}f_2$. Build it twice: once with the $f_1{+}f_2$
component in a fixed phase relation to the other two, as if generated from them, and once
with its phase drifting freely. The two signals contain the same frequencies at the same
strengths, so their power spectra are essentially identical. Only the bispectrum separates
them.

In [ ]:
# Build the two signals (plus a noise-only reference). Same three frequencies in both;
# only the phase behaviour of the f1 + f2 component differs.
f1, f2 = 1200.0, 520.0                          # both on exact 40 Hz STFT bins
t = np.arange(4 * SR) / SR                      # 4 s of signal
rng_demo = np.random.default_rng(1)

base = np.sin(2 * np.pi * f1 * t + 1.1) + np.sin(2 * np.pi * f2 * t + 0.4)
locked = base + np.sin(2 * np.pi * (f1 + f2) * t + 2.7)             # fixed phase relation
drift = np.cumsum(rng_demo.standard_normal(t.size)) * 0.05          # slowly wandering phase
unlocked = base + np.sin(2 * np.pi * (f1 + f2) * t + drift)
background = rng_demo.standard_normal(t.size)                       # no components at all

In [ ]:
# Measure each signal: the formula above, written out (bs.raw_bispectrum computes the
# same thing for every valid (f1, f2) pair at once; we use it from the next section on).
# Reported in signal-to-noise form, |B| / sqrt(P1 P2 P3): of order 1 when fully
# phase-locked, at the noise floor when not.
k1, k2, k3 = hz_to_bin(f1), hz_to_bin(f2), hz_to_bin(f1 + f2)
for name, sig in [('phase locked  ', locked), ('phase drifting', unlocked), ('noise only    ', background)]:
    Xs = bs.stft((sig + 0.1 * rng_demo.standard_normal(t.size)).astype(np.float32))
    P_demo = (np.abs(Xs) ** 2).mean(1)                              # power per bin
    B_cell = np.mean(np.real(Xs[k1] * Xs[k2] * np.conj(Xs[k3])))    # the triple product, averaged
    snr = abs(B_cell) / np.sqrt(P_demo[k1] * P_demo[k2] * P_demo[k3])
    print(f'{name}: power at f1+f2 = {P_demo[k3]:.2e}   coupling S/N = {snr:.2f}')

In [ ]:
# Listen: the difference is barely audible, yet the bispectrum separates them cleanly.
from IPython.display import Audio, display
display(Audio(locked, rate=SR), Audio(unlocked, rate=SR))

The phase-locked triplet reads close to 1 (fully coupled) while the drifting one sits at
the noise floor, despite carrying the same power at $f_1{+}f_2$. The gap grows with
averaging: uncoupled contributions cancel over frames, coupled ones add up. That was one
cell of the bispectrum; the rest of the notebook works with the whole $(f_1, f_2)$ plane.

### Coupling from a nonlinearity

Coupling rarely comes one frequency at a time. Next, a signal with coupling everywhere:
broadband noise `x` through the mild nonlinearity `y = x + A x²`, which couples every pair
of frequencies, and whose bispectrum we know exactly.

In [ ]:
# Broadband noise x with a smooth bump around 1.4 kHz. (The synthesis details do not
# matter; any wideband source would do.)
rng = np.random.default_rng(0)
nsamp = SR * 20                                              # 20 s clip: long, so averages are smooth
ff = np.fft.rfftfreq(nsamp, 1 / SR)                          # frequency of each FFT bin (Hz)
band = np.exp(-0.5 * ((ff - 1400) / 1100) ** 2) * (ff > 60)  # Gaussian mid-band mask, drops DC/low
x = np.fft.irfft(np.fft.rfft(rng.standard_normal(nsamp)) * np.sqrt(band), n=nsamp)  # coloured noise
x /= x.std()                                                 # normalise to unit variance

In [ ]:
# Now the coupling: push x through a mild quadratic nonlinearity, y = x + A x^2.
# The x^2 term creates sum tones f1+f2 that stay phase-locked to the pair (f1, f2) that made them,
# and that phase-locking is what the bispectrum detects.
A = 0.3                                                     # coupling strength (small = subtle)
y = (x + A * (x ** 2 - (x ** 2).mean())).astype(np.float32)  # mean subtracted

In [ ]:
# The raw waveform looks like noise: the coupling is completely invisible here.
ms = np.arange(int(0.03 * SR)) / SR * 1000                  # first 30 ms, in milliseconds
_, ax = plt.subplots(figsize=(8, 1.9), layout='constrained')
ax.plot(ms, y[:ms.size], color='0.15', lw=0.8); ax.set_xlim(0, 30)
ax.set_xlabel('time (ms)'); ax.set_ylabel('amplitude'); ax.set_title('Signal (30 ms)')
ax.tick_params(direction='in', top=True, right=True)
plt.show()

In [ ]:
# Compute the short-time Fourier transform (STFT), before the summary statistics.
X = bs.stft(y)                                  # complex STFT: one column per frame


In [ ]:
# Power spectrum: mean power per bin over the clip. Bin 0 (the mean) is never
# used below; it stays only so bin index = array index everywhere.
P = (np.abs(X) ** 2).mean(1)

In [ ]:
# Full bispectrum on the STFT's own 40 Hz bins, averaged over every frame:
#   B(f1, f2) = mean_frames Re( X(f1) X(f2) X*(f1+f2) )
# Flat form: one value per valid triplet (k2 <= k1, k1 + k2 inside the window; bin 0,
# the mean, is excluded since the bispectrum is defined for mean-free signals).
k1, k2, vals = bs.raw_bispectrum(X)
# Assemble the square picture for plotting: redundant half filled by the symmetry
# B(f1, f2) = B(f2, f1). raw_bispectrum(X, return_full=True) does both steps at once.
k, _, B = bs.full_bispectrum(k1, k2, vals)
f_grid = k * (SR / N_FFT)                      # the bin grid, in Hz (same for both axes)

In [ ]:
# Flat form vs assembled square:
vals.shape, B.shape

In [ ]:
# The signal-to-noise form divides out sqrt(P1 P2 P3) (inverse-variance weighting):
# loudness removed, the coupling itself remains. It reads as a deviation from
# Gaussianity: a purely Gaussian signal is fully described by its power spectrum, and
# this map would show only its noise floor: values of either sign, of order 1 for a
# single frame, shrinking like 1 / sqrt(n_frames) with averaging (about 0.02 for this
# 20 s clip).
SNR = bs.snr_bispectrum(B, P)                      # the controlled signal needs no power floor
norm = bs.symlog_norm(B, decades=3)                # colour scale, reused by the next figure too
FMIN, FMAX = f_grid[0], 4000.0                     # one frequency range for every panel

In [ ]:
# Left: the power spectrum is smooth and featureless. Middle: the bispectrum lights up
# where frequencies are coupled, though its scale still follows the raw power. Right:
# the signal-to-noise form shows the coupling alone.
_, ax = plt.subplots(1, 3, figsize=(13.5, 4.0), layout='constrained')
kp = (freqs >= FMIN) & (freqs <= FMAX)
ax[0].semilogy(freqs[kp], (P / P[kp].max())[kp], color='0.15', lw=1.0)
ax[0].set_xlim(FMIN, FMAX); ax[0].set_box_aspect(1)   # square panel, same axis handling
freq_axis(ax[0], 'x')
ax[0].set_xlabel('$f$ (kHz)'); ax[0].set_ylabel('$P(f)$ (normalized)'); ax[0].set_title('Power spectrum')
ax[0].tick_params(direction='in', top=True, right=True, which='both')
for axi, D, ttl, lbl, nrm in [
        (ax[1], B, 'Bispectrum $B(f_1,f_2)$', '$B$', norm),
        (ax[2], SNR, r'Signal-to-noise $B/\sqrt{P_1P_2P_3}$', r'$B/\sqrt{P_1P_2P_3}$',
         bs.symlog_norm(SNR, decades=2))]:
    bs.plot_bispectrum(f_grid, f_grid, D, ax=axi, cmap='RdBu_r', norm=nrm,
                       title=ttl, colorbar_label=lbl)
    axi.set_xlim(FMIN, FMAX); axi.set_ylim(FMIN, FMAX)
    freq_axis(axi, 'x'); freq_axis(axi, 'y')
    axi.set_xlabel('$f_1$ (kHz)'); axi.set_ylabel('$f_2$ (kHz)')
plt.show()

### The modal bispectrum estimator

The full estimate above is big and slow: ~10,000 triplet values per frame, at `O(F²)` cost,
each one a noisy single triple product. `mel_bispectrogram` uses the **modal estimator**
instead (see [the theory notes](../docs/theory.md)): the bispectrum expanded on a small basis
of smooth frequency modes, 49 coefficients by default, computed straight from the STFT
without ever forming the 2-D grid. To check what those coefficients capture, we rebuild
$B(f_1,f_2)$ from them and compare, both to the full estimate and to the analytic known answer.

In [ ]:
# The basis behind mel_bispectrogram: smooth frequency "modes" and their pairings.
# You never call these functions in day-to-day use (mel_bispectrogram wraps them);
# they appear here to check what the 49 coefficients capture.
modes = bs.mel_legendre_modes(12, sr=SR)          # degree 12: 13 modes, sampled on the STFT bins
pairs = bs.modal_index_pairs(12)                  # the 49 mode-pairs = the 49 coefficients
print('number of coefficients:', len(pairs))

In [ ]:
# Project the bispectrum onto the 49 pair kernels, then rebuild the 2-D picture from
# those 49 numbers alone.
beta = bs.project_bispectrum(X, pairs, modes)
# kmin=1 evaluates on the estimator's own bins, cell-for-cell aligned with B above.
_, _, Brec = bs.reconstruct_bispectrum(beta, pairs, modes,
                                       gram=bs.modal_gram_matrix(pairs, modes), kmin=1,
                                       return_full=True)

In [ ]:
# Known answer for y = x + A x^2:
#   B = 2A [ P(f1)P(f2) + P(f2)P(f3) + P(f1)P(f3) ],  with P the measured power spectrum.
G1, G2 = np.meshgrid(f_grid, f_grid, indexing='ij')
P1, P2, P3 = P[hz_to_bin(G1)], P[hz_to_bin(G2)], P[hz_to_bin(G1 + G2)]
Bth = np.where(np.isfinite(B), 2 * A * (P1 * P2 + P2 * P3 + P1 * P3), np.nan)

# The estimate carries an overall analysis-window constant, so scale theory to its amplitude.
mm = np.isfinite(B) & np.isfinite(Brec)
Bth *= np.median(B[mm] / Bth[mm])

In [ ]:
# One shared colour scale for the three panels.
_, ax = plt.subplots(1, 3, figsize=(13.5, 4.2), layout='constrained')
for axi, (D, ttl) in zip(ax, [(B, 'Raw bispectrum'), (Brec, 'Modal bispectrum (rebuilt from 49 coefficients)'),
                              (Bth, 'Known answer (theory)')]):
    bs.plot_bispectrum(f_grid, f_grid, D, ax=axi, cmap='RdBu_r', norm=norm, title=ttl, colorbar=False)
    axi.set_xlim(FMIN, FMAX); axi.set_ylim(FMIN, FMAX)
    freq_axis(axi, 'x'); freq_axis(axi, 'y')
    axi.set_xlabel('$f_1$ (kHz)'); axi.set_ylabel('$f_2$ (kHz)')
bs.symlog_colorbar(ax[-1].images[0], ax=ax[-1], label='$B$')
plt.show()

In [ ]:
# How well do the three agree? Pearson correlation coefficient over the valid region.
print(f'correlation(full estimate, rebuilt-from-49 modal): {np.corrcoef(B[mm], Brec[mm])[0, 1]:.2f}')
print(f'correlation(full estimate, known answer)         : {np.corrcoef(B[mm], Bth[mm])[0, 1]:.2f}')

The 49 coefficients rebuild the bispectrum closely and agree with the analytic answer: a compact
summary of the third-order structure. Computed per frame, like a spectrogram, they give a
"**bispectrogram**", a companion to the familiar mel power spectrogram, here with the same signed-log
compression a downstream model would see. This signal is stationary, so both features are constant
in time (+ statistical noise).

In [ ]:
# The feature as a model would consume it. eps is overridden here purely for display
# contrast (it sets where the signed-log compression starts biting; the default is fine
# for feeding models, just flat to look at).
Bmel = bs.mel_bispectrogram(y, sr=SR, eps=1e-4)  # (49, n_frames): the feature, signed-log compressed
Lmel = bs.mel_spectrogram(S=X, sr=SR)            # (80, n_frames): the mel power spectrogram (dB)
print('bispectrogram', Bmel.shape, '  mel power spectrogram', Lmel.shape)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 4.2), layout='constrained', sharex=True)
im0 = ax[0].imshow(Lmel, aspect='auto', origin='lower', cmap='magma')   # top: familiar mel spectrogram
ax[0].set_ylabel('mel band'); ax[0].set_title('Mel power spectrogram')
fig.colorbar(im0, ax=ax[0], label='dB')
# The coefficients are signed: diverging map, symmetric range.
vmax = np.abs(Bmel).max()
im1 = ax[1].imshow(Bmel, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax[1].set_ylabel('coefficient'); ax[1].set_xlabel('frame'); ax[1].set_title('Bispectrogram (mel_bispectrogram, 49-d)')
fig.colorbar(im1, ax=ax[1], label='signed log')
plt.show()

### Pooling over time

`time_pool` turns the per-frame coefficients into one vector per clip. The default order
(signed-log compress each frame, then average) suits non-stationary audio. To hunt weak
coupling that is steady over a long clip, average the raw coefficients first and compress
once: `mel_bispectrogram(..., log=False)`, `time_pool`, then `signed_log`.

## 2. Real sounds

Now six real recordings from librosa's example set: for each we compute the clip-averaged
bispectrum in its signal-to-noise form, $B/\sqrt{P_1 P_2 P_3}$, alongside the power spectrum.

In [ ]:
# Six real recordings from librosa's example set, each with a friendly label.
GALLERY = [('trumpet', 'trumpet'), ('pistachio', 'ragtime piano'), ('choice', 'drums + bass'),
           ('robin', 'birdsong'), ('libri1', 'speech'), ('sweetwaltz', 'jazz ensemble')]

# Frequency crop for the full bispectrum, shared by every panel (native 40 Hz bins).
GMIN, GMAX = 40.0, 4000.0

In [ ]:
# For each clip, one STFT feeds everything:
#  - Pc: clip-averaged power spectrum, one value per STFT bin
#  - Bc: clip-averaged bispectrum on the [GMIN, GMAX] window (all three legs band-limited),
#    as the square matrix (return_full=True), redundant half filled by symmetry
#  - the panel shows its signal-to-noise form, snr_bispectrum: loudness divided out so
#    coupling shows, with a power floor (floor=1e-3) since real spectra span many decades
panels = []
stfts = {}                                     # one STFT per clip, reused all section
for key, label in GALLERY:
    Sc = stfts[key] = bs.stft(load(key))       # complex STFT: one column per frame
    Pc = (np.abs(Sc) ** 2).mean(1)             # mean power per bin over the clip
    kk, _, Bc = bs.raw_bispectrum(Sc, kmin=hz_to_bin(GMIN), kmax=hz_to_bin(GMAX), return_full=True)
    gf1 = kk * (SR / N_FFT)                    # the cropped bin grid, in Hz
    panels.append((label, Pc, gf1, bs.snr_bispectrum(Bc, Pc, kmin=hz_to_bin(GMIN), floor=1e-3)))

In [ ]:
# One subfigure per clip: bispectrum on top, its power spectrum below, shared mel-warped x-axis.
keep = (freqs >= GMIN) & (freqs <= GMAX)

def clip_panel(sf, label, Pc, gf1, SNR):
    # 2 rows: square bispectrum (colorbar inset pinned to its box), thin power spectrum below.
    gs = sf.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.04)
    bis = sf.add_subplot(gs[0, 0]); psx = sf.add_subplot(gs[1, 0], sharex=bis)

    bs.plot_bispectrum(gf1, gf1, SNR, ax=bis, cmap='RdBu_r', norm=bs.symlog_norm(SNR, decades=3),
                       title=label, colorbar=False, aspect='auto', freq_scale='mel')
    bis.set_box_aspect(1)                                       # square bispectrum panel
    bis.set_xlabel(''); bis.tick_params(labelbottom=False)
    bis.set_ylabel('$f_2$ (kHz)'); freq_axis(bis, 'x'); freq_axis(bis, 'y')
    bs.symlog_colorbar(bis.collections[0], cax=bis.inset_axes([1.04, 0.0, 0.05, 1.0]),
                       label=r'$B/\sqrt{P_1P_2P_3}$')

    psx.semilogy(freqs[keep], (Pc / Pc[keep].max())[keep], color='0.15', lw=1.0)
    psx.set_box_aspect(1 / 3)                                   # same width as the square above
    bs.set_freq_scale(psx, 'mel', axis='x'); psx.set_xlim(GMIN, GMAX); freq_axis(psx, 'x')
    psx.set_xlabel('$f_1$ (kHz)'); psx.set_ylabel('$P(f_1)$ (normalized)')
    psx.tick_params(direction='in', top=True, right=True, which='both')

In [ ]:
fig = plt.figure(figsize=(14, 12), layout='constrained')
for sf, panel in zip(fig.subfigures(2, 3).ravel(), panels):
    clip_panel(sf, *panel)
plt.show()

### Do the sounds separate at the bispectrum level?

Correlate the clips pairwise. Shapes are compared on the signal-to-noise maps
(inverse-variance weighting, as above), once on the fine bin grid and once after
`mel_bin_bispectrum`, which bins both frequency axes onto 80 mel bands.

In [ ]:
# Per clip, full frequency range this time: power spectrum, bispectrum, and its
# signal-to-noise form, plus the mel-binned version of the latter.
raws = []
for key, label in GALLERY:
    Xc = stfts[key]                            # reuse the gallery STFT
    Pc = (np.abs(Xc) ** 2).mean(1)
    kf, _, Bc = bs.raw_bispectrum(Xc, return_full=True)
    Sc = bs.snr_bispectrum(Bc, Pc, floor=5e-2)     # heavier floor than the gallery: the full
    raws.append((key, label, Pc, kf * (SR / N_FFT), Bc, Sc))  # range includes the empty top octave
labels = [label for _, label, *_ in raws]
snr_mel_raw = [bs.mel_bin_bispectrum(Sc, sr=SR, n_fft=N_FFT) for *_, Sc in raws]

In [ ]:
# Helpers for the correlation matrices: cosine similarity between clips, and an
# annotated matrix plot (reused for the modal and band-basis versions below).
def cosine_matrix(maps):
    V = np.stack([m[np.isfinite(m)] for m in maps])    # every clip shares the same grid/mask
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    return V @ V.T

def plot_matrix(ax, C, title, ylabels=True):
    im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
    for a in range(len(labels)):
        for b in range(len(labels)):
            ax.text(b, a, f'{C[a, b]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(C[a, b]) > 0.6 else 'black')
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)), labels if ylabels else [''] * len(labels))
    ax.set_title(title)
    return im

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6), layout='constrained')
plot_matrix(ax[0], cosine_matrix([Sc for *_, Sc in raws]), 'Signal-to-noise bispectra')
im = plot_matrix(ax[1], cosine_matrix(snr_mel_raw), 'Same, mel-binned (80 bands)', ylabels=False)
fig.colorbar(im, ax=ax[-1], label='shape correlation')
plt.show()

The six sounds are nearly orthogonal at the bispectrum level: distinct coupling shapes,
even for clips whose power spectra overlap. Mel binning keeps that separation while
shrinking the grid from 200 x 200 = 40,000 values to 80 x 80 = 6,400.

### The 49-coefficient modal bispectrum feature

Same recipe as section 1, per clip: project each STFT onto the 49 mode-pairs and rebuild
$B(f_1, f_2)$ from those 49 numbers alone.

In [ ]:
# Rebuild per clip, reusing the raw estimates above and the section 1 basis
# (modes / pairs / Gram matrix, shared by every clip).
G = bs.modal_gram_matrix(pairs, modes)
recon = []
for key, label, Pc, fgc, Bc, Sc in raws:
    beta_c = bs.project_bispectrum(stfts[key], pairs, modes)   # the 49 coefficients
    _, _, Brc = bs.reconstruct_bispectrum(beta_c, pairs, modes, gram=G,
                                          kmin=1, return_full=True)
    recon.append((label, Brc, bs.snr_bispectrum(Brc, Pc, floor=5e-2)))

In [ ]:
# Top: the full bispectrum ("raw"); bottom: rebuilt from the coefficients, both in
# signal-to-noise form so one colour scale serves every panel (right). The figure is a
# helper because the band-basis section below draws the same one.
norm_g = bs.symlog_norm(np.concatenate([Sc[np.isfinite(Sc)] for *_, Sc in raws]), decades=3)

def rebuild_figure(rebuilds, tag):
    fig = plt.figure(figsize=(17, 5.6), layout='constrained')
    gs = fig.add_gridspec(2, 7, width_ratios=[1] * 6 + [0.08])
    ax = np.array([[fig.add_subplot(gs[i, j]) for j in range(6)] for i in range(2)])
    for j, ((key, label, Pc, fgc, Bc, Sc), (_, _, Sr)) in enumerate(zip(raws, rebuilds)):
        for i, (S, ttl) in enumerate([(Sc, f'{label}, raw'), (Sr, f'{label}, {tag}')]):
            axi = ax[i, j]
            bs.plot_bispectrum(fgc, fgc, S, ax=axi, cmap='RdBu_r', norm=norm_g,
                               title=ttl, colorbar=False, aspect='auto', freq_scale='mel')
            axi.set_xlim(GMIN, GMAX); axi.set_ylim(GMIN, GMAX); axi.set_box_aspect(1)
            freq_axis(axi, 'x'); freq_axis(axi, 'y')
            axi.set_xlabel('$f_1$ (kHz)' if i == 1 else ''); axi.set_ylabel('$f_2$ (kHz)' if j == 0 else '')
            if i == 0: axi.tick_params(labelbottom=False)
            if j > 0: axi.tick_params(labelleft=False)
    bs.symlog_colorbar(ax[0, 0].collections[0], cax=fig.add_subplot(gs[:, 6]),
                       label=r'$B/\sqrt{P_1P_2P_3}$')
    plt.show()

rebuild_figure(recon, 'modal')

In [ ]:
# Agreement between rebuild and full estimate, per clip: the plain correlation, and the
# harder version on the mel-binned signal-to-noise maps (pure coupling shape, compared at
# the resolution the basis lives on).
snr_mel_modal = [bs.mel_bin_bispectrum(Src, sr=SR, n_fft=N_FFT) for *_, Src in recon]

print(f"{'clip':16s}{'plain r':>9s}{'S/N mel-binned r':>18s}")
for (key, label, Pc, fgc, Bc, Sc), (_, Brc, Src), Smr, Smm in zip(raws, recon, snr_mel_raw, snr_mel_modal):
    m = np.isfinite(Bc) & np.isfinite(Brc)
    mm = np.isfinite(Smr) & np.isfinite(Smm)
    print(f'{label:16s}{np.corrcoef(Bc[m], Brc[m])[0, 1]:>9.2f}'
          f'{np.corrcoef(Smr[mm], Smm[mm])[0, 1]:>18.2f}')

The rebuild visibly differs from its raw counterpart. By construction the modal bispectrum
is a projection onto a small basis of smooth functions, so sharp features (the specific
frequencies of some instruments) are not fully captured, and smooth artefacts appear in
their place; the smooth bispectrum of section 1 was rebuilt much more faithfully. 

In [ ]:
# Cross-clip matrix of the modal rebuilds (signal-to-noise form).
fig, ax = plt.subplots(figsize=(5.2, 4.6), layout='constrained')
im = plot_matrix(ax, cosine_matrix([Src for *_, Src in recon]), 'Modal rebuilds (signal-to-noise)')
fig.colorbar(im, ax=ax, label='shape correlation')
plt.show()

The between-sound structure of the raw maps survives in the rebuilds: 49 numbers per frame
still separate the six sounds, with an increased correlation between drum+bass and speech.

### Alternative: the 210-coefficient mel-band bispectrum

`basis="bands"` swaps the smooth Legendre modes for the mel bands themselves: at 20 bands,
210 localized coefficients instead of 49 smooth global ones. 

In [ ]:
# Same project / rebuild loop with the mel-band basis (20 bands = 210 coefficients).
mband = bs.mel_band_modal_bispectrum(20, sr=SR)
Gb = bs.modal_gram_matrix(mband.pairs, mband.modes)
recon_bands = []
for key, label, Pc, fgc, Bc, Sc in raws:
    beta_b = bs.project_bispectrum(stfts[key], mband.pairs, mband.modes)
    _, _, Brb = bs.reconstruct_bispectrum(beta_b, mband.pairs, mband.modes, gram=Gb,
                                          kmin=1, return_full=True)
    recon_bands.append((label, Brb, bs.snr_bispectrum(Brb, Pc, floor=5e-2)))

In [ ]:
# Same figure as before: full estimate on top, band-basis rebuild below, on the shared
# colour scale.
rebuild_figure(recon_bands, 'bands')

In [ ]:
# Fidelity to the full estimate, side by side: the 49 smooth Legendre coefficients
# against the 210 localized band coefficients.
print(f"{'clip':16s}{'plain r, Legendre (49)':>24s}{'plain r, bands (210)':>22s}")
for (key, label, Pc, fgc, Bc, Sc), (_, Brc, _), (_, Brb, _) in zip(raws, recon, recon_bands):
    ml = np.isfinite(Bc) & np.isfinite(Brc)
    m = np.isfinite(Bc) & np.isfinite(Brb)
    print(f'{label:16s}{np.corrcoef(Bc[ml], Brc[ml])[0, 1]:>24.2f}'
          f'{np.corrcoef(Bc[m], Brb[m])[0, 1]:>22.2f}')

In [ ]:
# Cross-clip matrix of the band-basis rebuilds (signal-to-noise form).
fig, ax = plt.subplots(figsize=(5.2, 4.6), layout='constrained')
im = plot_matrix(ax, cosine_matrix([Srb for *_, Srb in recon_bands]), 'Band-basis rebuilds (signal-to-noise)')
fig.colorbar(im, ax=ax, label='shape correlation')
plt.show()

The localized bands track the raw bispectrum more closely on every clip, at the price of
a much larger vector: 210 coefficients per frame instead of 49, with no smooth global
structure to truncate by degree.

### The bispectrum over time

Everything above is clip-averaged, but the estimate exists per frame
(`raw_bispectrum(X, average=False, return_full=True)`), and `bs.animate_bispectrum` plays
it as a movie. Below, ~5 s of solo trumpet: the harmonic lattice moves with the melody.
The cell generates the gif with [`animate_bispectrum.py`](animate_bispectrum.py) on first
run (about a minute), then displays it.

In [ ]:
# Generate the animation on first run (the gif is not stored in git), then display it.
import subprocess, sys
from pathlib import Path
from IPython.display import Image

if not Path('animate_bispectrum.gif').exists():
    subprocess.run([sys.executable, 'animate_bispectrum.py'], check=True)
Image('animate_bispectrum.gif')

### Cost: the raw grid vs the modal bases

What the modal estimator buys is a small per-frame vector now, and speed that scales when
the frequency resolution grows. Every timing below covers the whole call from the
waveform, STFT included, with the mel power spectrogram as the familiar reference point.

In [ ]:
import time

def bench(fn, n=5):
    # best of n runs: outside interference only ever adds time (same idea as timeit)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return min(ts)

In [ ]:
# One 4 s clip through each feature. The tag says what shape the call returns: the raw
# grid is clip-averaged by default, the others keep one vector per frame.
yb = load('trumpet')
runs = [
    ('mel power spectrogram', lambda: bs.mel_spectrogram(yb, sr=SR),                             'per frame'),
    ('mel Legendre (deg 12)', lambda: bs.mel_bispectrogram(yb, sr=SR),                           'per frame'),
    ('mel bands (20 bands)',  lambda: bs.mel_bispectrogram(yb, sr=SR, basis='bands', n_mels=20), 'per frame'),
    ('raw grid',              lambda: bs.raw_bispectrum(bs.stft(yb)),                            'per clip'),
]
print(f"{'feature':25s}{'time':>8s}{'output size':>14s}")
for name, fn, per in runs:
    out = fn()
    size = out[2].size if isinstance(out, tuple) else out.shape[0]
    print(f'{name:25s}{bench(fn) * 1e3:>5.0f} ms{size:>10d} {per}')

In [ ]:
# At the default frame length (n_fft=400) all computations are cheap. 
# The raw grid costs O(F^2) per frame, the modal estimators close to O(F), so
# doubling the frequency resolution widens the gap roughly 2x each time.
for n_fft in (400, 800, 1600):
    times = [bench(lambda: fn(n_fft)) * 1e3 for fn in (
        lambda n: bs.mel_spectrogram(yb, sr=SR, n_fft=n),
        lambda n: bs.mel_bispectrogram(yb, sr=SR, n_fft=n),
        lambda n: bs.mel_bispectrogram(yb, sr=SR, n_fft=n, basis='bands', n_mels=20),
        lambda n: bs.raw_bispectrum(bs.stft(yb, n_fft=n)))]
    print(f'n_fft={n_fft:5d}: log-mel {times[0]:4.0f} ms, mel Legendre {times[1]:4.0f} ms, '
          f'mel bands {times[2]:4.0f} ms, raw {times[3]:5.0f} ms')

## What to use

- `bs.mel_bispectrogram(y, sr)`: the feature. 49 signed coefficients per frame;
  `bs.time_pool` turns them into one vector per clip.
- Pooling order: default (compress, then average) for real audio; average raw coefficients
  first (`log=False`, `time_pool`, `signed_log`) to hunt weak steady coupling.
- `bs.raw_bispectrum(X)` and `bs.full_bispectrum(k1, k2, values)`: the explicit estimate
  and its square picture, drawn with `bs.plot_bispectrum`.
- `bs.animate_bispectrum(...)`: the per-frame estimate as a movie (see
  `examples/animate_bispectrum.py`).
- `bs.snr_bispectrum(B, P)`: the signal-to-noise form, for display and shape comparison.
- `bs.mel_bin_bispectrum(B, sr=sr)`: the bispectrum at mel resolution.